<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/04%20-%20Bayesian%20Optimization/04a_gaussian_processes.ipynb)

# 04a · 贝叶斯优化（上）：从零理解高斯过程与预测不确定性

> 对应林雪平大学（LiU）课程 `Lecture 08 - Bayesian learning`。本节我们将把原课程的正态分布推导与有机半导体薄膜退火实验、分子溶解度预测结合起来，用纯 NumPy 手写一遍高斯过程回归（GPR），彻底打破黑箱。

在前面 `02` 和 `03` 讲里，无论是线性回归、随机森林还是聚类，都有一个共同前提：**数据很多**（AqSolDB 有近 1 万个分子）。但在真实的材料实验室里，情况往往正相反：
- 合成一个新配体、或者旋涂/退火测试一片有机薄膜器件，往往要花 **1 天到 1 周**；
- 你手里可能只有 **5 到 15 个实验数据点**；
- 你不仅想知道模型预测哪个条件性能最好，更想知道**模型对哪块区域"心里没底"（不确定性有多大）**。

1. 为什么小样本实验需要"不确定性"？
2. 从一维正态分布到二维相关高斯分布
3. 核函数（Kernel）：定义"多近才算像"与先验采样
4. 仅用 10 行 NumPy 手写高斯过程回归（GPR）
5. 用 `scikit-learn` 处理带实验噪声的数据
6. 离散分子怎么算相似性？—— Tanimoto 核与溶解度不确定性验证

### 这个 notebook 怎么用

| 标记 | 意思 | 你要做什么 |
|---|---|---|
| 📘 | 讲解 | 读一读 |
| 🤔 | 先猜一猜 | **运行之前**先想想会输出什么 |
| ▶️ | 运行看看 | 选中代码格，按 `Shift + Enter` |
| ✏️ | 改一改 | 改代码里标了 `# ✏️` 的地方，再运行 |
| 🏋️ | 练习 | 自己写代码；参考答案在 `solutions/` 文件夹 |
| 💡 | 小贴士 | 有用但不是必须记住的东西 |

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "04 - Bayesian Optimization"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from sklearn.base import clone
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

In [ ]:
# 本地运行：文件在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮（本节需要 solubility.csv）
def find_file(name):
    for path in (f"data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请把它放在 data/ 文件夹里")

---
## 1. 为什么材料实验需要"不确定性"？

📘 在 `02a` 里，我们看到随机森林对没见过的并五苯给出了一个确切的预测值，但它**没有告诉你这个预测非常不靠谱**（直到我们在 `03a` 用最近邻距离检查适用域才发现问题）。

在材料工艺优化中，假设我们在研究**有机半导体薄膜的退火温度 $T$（$60^\circ\text{C} \sim 200^\circ\text{C}$）与载流子迁移率 $\mu$** 的关系。温度太低结晶度不够，温度太高薄膜会开裂或发生相分离。
- 我们先在实验室做了 6 组不同温度的退火测试，但中间有很大一段温度区间还没测过；
- **高斯过程回归（Gaussian Process Regression, GPR）** 不只输出一条预测曲线 $\mu(x)$，还会输出每个点上的**标准差 $\sigma(x)$**（置信区间）——在测过的点附近 $\sigma$ 很小，在没测过的空白区间 $\sigma$ 会鼓起来，诚实地告诉你："这里我没见过，别全信我！"

---
## 2. 从单变量高斯到多维相关高斯

📘 一维正态分布 $y \sim \mathcal{N}(\mu, \sigma^2)$ 大家都很熟。如果我们有**两个**实验点 $y_1, y_2$，它们服从二维联合正态分布：
$$\begin{bmatrix} y_1 \\ y_2 \end{bmatrix} \sim \mathcal{N}\left(\begin{bmatrix} 0 \\ 0 \end{bmatrix}, \begin{bmatrix} 1 & \rho \\ \rho & 1 \end{bmatrix}\right)$$
这里的 $\rho$ 就是协方差（相关系数）。**高斯过程的核心思想非常简单：** 如果两个实验条件 $x_1$ 和 $x_2$ 很接近（比如退火温度只差 $2^\circ\text{C}$），那么测出来的材料性能 $y_1$ 和 $y_2$ 就应该高度正相关（$\rho \approx 1$）；如果差了 $100^\circ\text{C}$，它们就几乎不相关（$\rho \approx 0$）。

▶️ 运行下面这格，直观感受 $\rho = 0.9$ 和 $\rho = 0.1$ 时采样出的 $(y_1, y_2)$ 点对。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, rho in zip(axes, [0.9, 0.1]):
    cov = [[1.0, rho], [rho, 1.0]]
    samples = np.random.multivariate_normal([0, 0], cov, size=300)
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.45, s=18)
    ax.set_title(f"rho = {rho} (correlation)")
    ax.set_xlabel("y1 at x1")
    ax.set_ylabel("y2 at x2")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. 核函数（Kernel）：从"几个点"变成"连续函数"

📘 现在把二维推广到 $N=100$ 维：我们在横轴上密密麻麻取 100 个温度点 $x_1, x_2, \dots, x_{100}$。怎么决定任意两点 $x_i$ 和 $x_j$ 的相关性 $\Sigma_{ij}$ 呢？这就靠**核函数（Kernel）**。

最经典的是 **RBF（径向基 / 高斯）核函数**：
$$k(x_i, x_j) = \sigma_f^2 \exp\left(-\frac{(x_i - x_j)^2}{2\ell^2}\right)$$
- **距离 $|x_i - x_j|$ 越近**，指数项越接近 $1$，说明两点的函数值几乎同步变化；
- **长度尺度 $\ell$（`length_scale`）**：控制函数的"变化快慢"。$\ell$ 越大，相隔很远的点依然相关，曲线越平缓；$\ell$ 越小，曲线越曲折震荡。

▶️ 下面我们不看任何真实数据，仅仅给定核函数，从**高斯过程先验（Prior）**中随机抽取 5 条光滑曲线。

In [ ]:
def rbf_kernel(X1, X2, length_scale=1.0, sigma_f=1.0):
    # X1: (n1, 1), X2: (n2, 1)
    sq_dist = np.sum(X1**2, axis=1, keepdims=True) + np.sum(X2**2, axis=1) - 2 * X1 @ X2.T
    return (sigma_f ** 2) * np.exp(-0.5 * sq_dist / (length_scale ** 2))

X_grid = np.linspace(0, 10, 100).reshape(-1, 1)

# ✏️ 试着把 length_scale 改成 0.3 或 3.0，再运行看看！
length_scale = 1.2
K_prior = rbf_kernel(X_grid, X_grid, length_scale=length_scale) + 1e-8 * np.eye(len(X_grid))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(K_prior, cmap="viridis", origin="lower")
axes[0].set_title(f"Prior Covariance Matrix K (l={length_scale})")
axes[0].set_xlabel("Grid index j")
axes[0].set_ylabel("Grid index i")
plt.colorbar(im, ax=axes[0], shrink=0.8)

# 从 100 维联合正态分布采样 5 条曲线
prior_samples = np.random.multivariate_normal(np.zeros(len(X_grid)), K_prior, size=5)
for s in prior_samples:
    axes[1].plot(X_grid.ravel(), s, lw=1.8, alpha=0.85)
axes[1].set_title(f"5 Random Functions Drawn from Prior (l={length_scale})")
axes[1].set_xlabel("x")
axes[1].set_ylabel("f(x)")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

📘 看左边的协方差矩阵：对角线最亮（一个点和自己相关系数为 1），离开对角线越远越暗（距离远了就不相关了）。这 5 条随机曲线，就是模型在**还没看到任何实验数据之前**对世界的所有"合理想象"。

---
## 4. 仅用 10 行 NumPy 手写高斯过程回归（GPR）

📘 一旦我们在实验室测出了 $n$ 个点 $(X_{\text{train}}, y_{\text{train}})$，那些**不经过这些实验点**的先验曲线就被淘汰了，剩下的曲线构成了**后验分布（Posterior）**。

利用多元高斯分布的条件概率公式，给定观测值 $y$（带观测噪声方差 $\sigma_n^2$），任意新位置 $X_*$ 的预测均值 $\mu_*$ 和协方差 $\Sigma_*$ 有精确的解析解：
$$\mu_* = K_*^T (K + \sigma_n^2 I)^{-1} y$$
$$\Sigma_* = K_{**} - K_*^T (K + \sigma_n^2 I)^{-1} K_*$$
其中 $K = k(X_{\text{train}}, X_{\text{train}})$，$K_* = k(X_{\text{train}}, X_*)$，$K_{**} = k(X_*, X_*)$。

🤔 **先猜一猜：** 如果设噪声 $\sigma_n = 0$，在观测点 $X_{\text{train}}$ 的正上方，后验标准差 $\sigma_*$ 会是多少？在远离观测点的地方 $\sigma_*$ 又会怎样？

▶️ 运行下面的代码，用 10 行 NumPy 拟合一组模拟的"有机薄膜退火温度 $T$ vs 迁移率 $\mu$"实验数据。

In [ ]:
# 模拟实验室测得的 6 个退火温度点（x 已缩放到 0~10 区间，代表 60~200°C 的归一化坐标）
X_train = np.array([1.0, 2.2, 3.5, 7.2, 8.5, 9.3]).reshape(-1, 1)
def true_mobility(x):
    return np.sin(0.9 * x) + 0.3 * np.cos(2.1 * x)

y_train = true_mobility(X_train).ravel()

def gp_posterior(X_tr, y_tr, X_te, length_scale=1.2, sigma_f=1.0, noise=1e-4):
    K = rbf_kernel(X_tr, X_tr, length_scale, sigma_f) + noise * np.eye(len(X_tr))
    K_s = rbf_kernel(X_tr, X_te, length_scale, sigma_f)
    K_ss = rbf_kernel(X_te, X_te, length_scale, sigma_f) + 1e-8 * np.eye(len(X_te))
    
    K_inv = np.linalg.inv(K)
    mu_s = K_s.T @ K_inv @ y_tr
    cov_s = K_ss - K_s.T @ K_inv @ K_s
    return mu_s, cov_s

mu_s, cov_s = gp_posterior(X_train, y_train, X_grid, length_scale=1.2, noise=1e-4)
std_s = np.sqrt(np.maximum(np.diag(cov_s), 0))

plt.figure(figsize=(9, 4.5))
plt.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5, label="True unknown curve")
plt.plot(X_grid, mu_s, "b-", lw=2, label="GP posterior mean")
plt.fill_between(X_grid.ravel(), mu_s - 1.96 * std_s, mu_s + 1.96 * std_s,
                 color="blue", alpha=0.18, label="95% confidence interval")
# 从后验分布也抽 3 条可能曲线
post_samples = np.random.multivariate_normal(mu_s, cov_s, size=3)
for i, s in enumerate(post_samples):
    plt.plot(X_grid.ravel(), s, lw=1, alpha=0.6, linestyle=":", label="Posterior sample" if i == 0 else None)

plt.scatter(X_train, y_train, c="red", s=60, zorder=5, label="Lab measurements (n=6)")
plt.title("Hand-written GPR on Annealing Temperature vs Mobility")
plt.xlabel("Normalized Annealing Temperature")
plt.ylabel("Normalized Mobility")
plt.legend(loc="lower left", fontsize=9)
plt.grid(alpha=0.3)
plt.show()

📘 注意看图上 $x \in [4.0, 6.5]$ 这片完全没有实验数据的空白区：蓝色阴影（95% 置信区间）明显膨胀鼓起！GPR 非常诚实地告诉我们："这中间我不知道，真实曲线可能是虚线里的任何一条"。而下一讲（`04b`）的**贝叶斯优化**，正是利用这块鼓起来的"高不确定性区域"来决定下一个实验应该测哪里！

---
## 5. `scikit-learn` 实战：自动学超参数与实验噪声

📘 手写公式时，我们人为猜了 `length_scale=1.2`。但在实际科研中，我们并不知道 $\ell$ 和测量噪声 $\sigma_n$ 应该是多少。`sklearn.gaussian_process.GaussianProcessRegressor` 会在调用 `.fit()` 时，通过**最大化边缘对数似然（Log Marginal Likelihood）**自动把最优的 $\ell$、振幅和噪声学出来。

▶️ 运行下面这格，在实验数据上人为加入 $\pm 0.15$ 的随机测量误差，对比"无噪声硬插值"和"带 `WhiteKernel` 噪声平滑"的区别。

In [ ]:
rng = np.random.RandomState(7)
y_noisy = y_train + rng.normal(0, 0.15, size=len(y_train))

# 1. 不带噪声核：强行穿过每一个带噪点（过拟合）
gp_no_noise = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-6, random_state=0)
gp_no_noise.fit(X_train, y_noisy)

# 2. 带 WhiteKernel：允许模型估计实验误差
kernel_noisy = C(1.0) * RBF(1.0) + WhiteKernel(noise_level=0.1)
gp_with_noise = GaussianProcessRegressor(kernel=kernel_noisy, n_restarts_optimizer=10, random_state=0)
gp_with_noise.fit(X_train, y_noisy)

print("无噪声核学到的参数:", gp_no_noise.kernel_)
print("带噪声核学到的参数:", gp_with_noise.kernel_)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, model, title in zip(axes, [gp_no_noise, gp_with_noise],
                            ["Without WhiteKernel (Overfits noise)", "With WhiteKernel (Smooth & Honest)"]):
    mu, std = model.predict(X_grid, return_std=True)
    ax.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5, label="True curve")
    ax.plot(X_grid, mu, "b-", lw=2, label="Mean")
    ax.fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.18, label="95% CI")
    ax.scatter(X_train, y_noisy, c="red", s=55, zorder=5, label="Noisy lab data")
    ax.set_title(title)
    ax.set_ylim(-2.2, 2.2)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 6. 分子没有连续坐标怎么办？—— Tanimoto 核函数

📘 到目前为止，输入 $x$ 都是连续的一维数字（如温度）。但如果我想用 GPR 预测**分子的溶解度 `logS`**（比如回到我们熟悉的 `solubility.csv`），分子是用 **2048 维 Morgan 指纹**表示的！对 0/1 向量算欧氏距离往往不如用化学界最经典的 **Tanimoto 相似度**（`01d` 讲过）：
$$K_{\text{Tanimoto}}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|^2 + \|\mathbf{b}\|^2 - \mathbf{a} \cdot \mathbf{b}}$$
好消息是：Tanimoto 相似度在数学上是一个**合法的正定核函数**，可以直接当成高斯过程的协方差矩阵 $K$！

🤔 **先猜一猜：** 如果我们在 400 个 AqSolDB 分子上用 Tanimoto 核训练 GPR，再去预测另外 200 个新分子：那些被模型打上"高不确定性 $\sigma$"的分子，它们的真实误差 $|\hat{y} - y|$ 会更大吗？

▶️ 运行这格验证一下 GPR 的"自知之明"。

In [ ]:
df = pd.read_csv(find_file("solubility.csv"))
# 过滤合法 SMILES 并随机抽取 600 个分子演示（400 训练 + 200 测试，CPU 只需 1 秒）
sub = df.sample(700, random_state=42).reset_index(drop=True)
fps, ys, valid_smiles = [], [], []
for smi, logs in zip(sub["SMILES"], sub["logS"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
        fps.append(np.array(fp, dtype=float))
        ys.append(logs)
        valid_smiles.append(smi)
    if len(fps) == 600:
        break

X_fp = np.array(fps)
y_logs = np.array(ys)
X_tr_fp, X_te_fp = X_fp[:400], X_fp[400:]
y_tr_fp, y_te_fp = y_logs[:400], y_logs[400:]

def tanimoto_kernel(A, B):
    dot = A @ B.T
    norm_a = np.sum(A**2, axis=1, keepdims=True)
    norm_b = np.sum(B**2, axis=1, keepdims=True).T
    denom = norm_a + norm_b - dot
    return np.where(denom > 0, dot / denom, 0.0)

# 用预先算好的核矩阵手写 GPR 预测
noise_var = 0.15
K_tr = tanimoto_kernel(X_tr_fp, X_tr_fp) + noise_var * np.eye(len(X_tr_fp))
K_s = tanimoto_kernel(X_tr_fp, X_te_fp)
K_ss_diag = np.ones(len(X_te_fp))  # 自己和自己的 Tanimoto 相似度为 1

# 用训练集的均值做中心化（先验均值为 0，所以减去训练集均值）
y_mean = y_tr_fp.mean()
K_inv = np.linalg.inv(K_tr)
mu_te = y_mean + K_s.T @ K_inv @ (y_tr_fp - y_mean)
var_te = K_ss_diag - np.sum((K_s.T @ K_inv) * K_s.T, axis=1)
std_te = np.sqrt(np.maximum(var_te, 0))

abs_err = np.abs(mu_te - y_te_fp)
rmse = np.sqrt(np.mean((mu_te - y_te_fp)**2))
print(f"400 个分子训练的 Tanimoto-GPR 测试集 RMSE = {rmse:.2f}")

# 按预测不确定性分成低不确定性（前 50%）和高不确定性（后 50%）两组
median_std = np.median(std_te)
low_unc_mae = abs_err[std_te <= median_std].mean()
high_unc_mae = abs_err[std_te > median_std].mean()
print(f"最有把握的一半分子 MAE = {low_unc_mae:.2f} | 最没把握的一半分子 MAE = {high_unc_mae:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
sc = axes[0].scatter(y_te_fp, mu_te, c=std_te, cmap="viridis", alpha=0.75, s=28)
axes[0].plot([-10, 2], [-10, 2], "r--", lw=1)
axes[0].set_xlabel("True logS")
axes[0].set_ylabel("Predicted logS")
axes[0].set_title("Parity Plot Colored by Uncertainty (std)")
plt.colorbar(sc, ax=axes[0], label="Predicted std")

axes[1].scatter(std_te, abs_err, alpha=0.65, s=25)
axes[1].axvline(median_std, color="red", linestyle="--", label=f"Median std={median_std:.2f}")
axes[1].set_xlabel("Predicted Uncertainty (std)")
axes[1].set_ylabel("Absolute Error |pred - true|")
axes[1].set_title("Does Higher Uncertainty Mean Larger Error?")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

💡 **看到了吗？** 模型对"最有把握的一半分子"平均误差明显更低，对"最没把握的一半分子"平均误差明显更高！这就是高斯过程在科研里的最大价值——**它自带适用域警报**，能告诉你哪些分子的预测不能轻信。

---
## 🏋️ 练习

### 练习 04a-1：比较不同核函数的光滑度
在 `sklearn` 里，除了最光滑的 `RBF()`，工程和物理中最常用的是 `Matern(nu=0.5)`（粗糙、不可导，像布朗运动）和 `Matern(nu=1.5)`（一阶可导）。
用第 5 节的带噪数据 `(X_train, y_noisy)`，分别用 `C(1.0)*Matern(1.0, nu=0.5) + WhiteKernel(0.1)` 和 `C(1.0)*Matern(1.0, nu=1.5) + WhiteKernel(0.1)` 拟合，把它们的均值和 95% 置信区间画在一张图上对比。

In [ ]:
# 在这里写你的代码

### 练习 04a-2：实验点数量如何改变不确定性？
在第 4 节的无噪声手写 `gp_posterior` 函数里，把训练点增加到 12 个（比如在空白区间 `[4.0, 6.5]` 多补测 6 个点），对比 `n=6` 和 `n=12` 时在整个区间上的平均不确定性 `np.mean(std_s)` 下降了多少？画图展示。

In [ ]:
# 在这里写你的代码

### 练习 04a-3：预测有机半导体分子的溶解度及不确定性
用第 6 节训练好的 `Tanimoto-GPR`（`K_inv` 和 `y_tr_fp`），计算下面 5 个有机半导体相关分子的预测 `logS` 和预测不确定性 `std`：
- 苯 `c1ccccc1`
- 噻吩 `c1ccsc1`
- 蒽 `c1ccc2cc3ccccc3cc2c1`
- 并四苯 `c1ccc2cc3cc4ccccc4cc3cc2c1`
- C8-BTBT `CCCCCCCCc1ccc2c(c1)sc1c2sc2cc(CCCCCCCC)ccc21`

🤔 哪个分子的预测不确定性 $\sigma$ 最小？哪个最大？为什么？（提示：和它们在 400 个训练集里的最大 Tanimoto 相似度有关！）

In [ ]:
# 在这里写你的代码